In [ ]:
# =============================================================================
# CELL 1 — Mount Drive and set working directory
# =============================================================================
from google.colab import drive
import os

drive.mount('/content/drive', force_remount=True)

PROJECT = '/content/drive/MyDrive/CASS_QESC'  # adjust if your folder name differs
os.chdir(PROJECT)

print("Working directory:", os.getcwd())
print("Files found:", os.listdir('.'))


In [ ]:
# =============================================================================
# CELL 2 — Install dependencies
# =============================================================================
# Run this once per session — Colab forgets installations on disconnect
!pip install sentence-transformers numpy -q

In [ ]:
# ── Regenerate embeddings for updated corpus ──────────────────────────────
import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open('QESC_v1.0.json', encoding='utf-8') as f:
    corpus = json.load(f)

model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

situations = [e['situation_en'] for e in corpus]

print(f"Encoding {len(situations)} entries...")
embeddings = model.encode(
    situations,
    batch_size=32,
    show_progress_bar=True,
    normalize_embeddings=True,
    convert_to_numpy=True
)

np.save('corpus_embeddings.npy', embeddings)
print(f"Saved. Shape: {embeddings.shape}")

In [ ]:
# =============================================================================
# CELL 3 — Load corpus, embeddings, and model
# =============================================================================
# This cell loads the three objects all subsequent cells depend on.
# corpus              — 68 prophet entries from QESC_v1.0.json
# corpus_embeddings   — pre-computed embedding matrix (68, 384)
# model               — MiniLM sentence encoder
#
# Expected output:
#   Corpus: 68 entries
#   Embeddings: (68, 384)

import json
import numpy as np
from sentence_transformers import SentenceTransformer

with open('QESC_v1.0.json', encoding='utf-8') as f:
    corpus = json.load(f)

corpus_embeddings = np.load('corpus_embeddings.npy')
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print(f"Corpus: {len(corpus)} entries")
print(f"Embeddings: {corpus_embeddings.shape}")


# =============================================================================
# CELL 4 — Constraint Functions v4 (Gaussian)
# =============================================================================
# C_AEA: Age-Emotional Alignment
#   Penalises mismatch between emotional intensity of query and corpus entry.
#   Uses Gaussian decay — smooth, no sharp boundaries.
#   lam=0.2: gentle penalty, allows nearby EI values to score well.
#
# C_RES: Narrative Resolution Alignment
#   Penalises mismatch between resolution type of query and corpus entry.
#   mu=2.0: strict — opposite RT scores near zero.

import numpy as np

def C_AEA(ei_query: float, ei_corpus: float, lam: float = 0.2) -> float:
    return float(np.exp(-lam * (ei_query - ei_corpus) ** 2))


def C_RES(rt_query: float, rt_corpus: float, mu: float = 2.0) -> float:
    return float(np.exp(-mu * (rt_query - rt_corpus) ** 2))

print("C_AEA and C_RES defined.")


# =============================================================================
# CELL 5 — EI and RT Detection Functions (v2)
# =============================================================================
# detect_EI(text) → float in [1.0, 5.0]
#   1 = mild, 3 = moderate, 5 = severe
#
# detect_RT(text) → float in {1.0, 2.0, 3.0}
#   1 = immediate (right now), 2 = gradual, 3 = chronic/delayed
#
# Supports English, French, and Darija.

def detect_EI(text: str) -> float:
    text_lower = text.lower()
    score = 2.0

    severe = [
        "want to die", "want to disappear", "kill myself", "end it all",
        "can't go on", "no reason to live", "hate myself", "hate my life",
        "worst day of my life", "i give up on everything", "nobody loves me",
        "completely alone in the world", "nobody cares if i exist"
    ]
    high = [
        "terrified", "devastated", "heartbroken", "shattered", "hopeless",
        "despair", "can't breathe", "i hate", "abandoned", "betrayed",
        "humiliated", "destroyed", "unbearable", "nightmare", "screaming",
        "can't stop crying", "sobbing", "so scared", "really scared",
        "very scared", "so angry", "really angry", "i was hurt badly",
        "deeply hurt", "so ashamed", "so embarrassed", "totally alone",
        "hits me", "hurts me", "beats me", "told everyone", "told my secret",
        "spread rumours", "spread rumors", "everyone hates me",
        "complètement seul", "je déteste", "je me déteste", "désespéré",
        "tellement peur", "j'ai tellement pleuré",
        "mkanch", "bgha imout", "ma bghitch n3ich", "khasni nmout"
    ]
    moderate = [
        "sad", "scared", "angry", "hurt", "worried", "upset", "nervous",
        "lonely", "left out", "ignored", "unfair", "crying", "cried",
        "ashamed", "embarrassed", "afraid", "anxious", "stressed",
        "disappointed", "frustrated", "confused", "lost", "helpless",
        "nobody understands", "no one listens", "no one cares",
        "feel bad", "feeling bad", "feel terrible", "feel awful",
        "feel horrible", "i don't know what to do",
        "triste", "peur", "seul", "personne ne comprend", "pas compris",
        "je pleure", "abandonné", "blessé", "inquiet",
        "mzyan", "3ayyan", "wahid", "ma3ndich", "yfahmni", "kaynch",
        "bkit", "khayf", "wahcha", "zhar", "mfhomch"
    ]
    low = [
        "a little", "kind of", "sort of", "a bit", "slightly", "maybe",
        "not really", "not that bad", "just a small", "tiny bit",
        "not so bad", "could be worse", "it's okay", "not too bad",
        "un peu", "pas vraiment", "pas trop grave",
        "shwiya", "mashi bzaf"
    ]
    intensifiers = [
        "very", "really", "so", "extremely", "totally",
        "completely", "absolutely", "deeply", "tellement", "bzaf"
    ]

    for m in severe:
        if m in text_lower:
            score += 2.0
    for m in high:
        if m in text_lower:
            score += 1.0
    for m in moderate:
        if m in text_lower:
            score += 0.5
    for m in low:
        if m in text_lower:
            score -= 0.5

    intensifier_count = sum(1 for w in intensifiers if w in text_lower)
    score += intensifier_count * 0.3
    #score = round(score * 2) / 2 the rounding is removed
    return float(min(max(score, 1.0), 5.0))


def detect_RT(text: str) -> float:
    text_lower = text.lower()
    immediate_score = 0
    chronic_score   = 0

    immediate_markers = [
        "right now", "right this moment", "this moment", "at this moment",
        "just happened", "just now", "today", "this morning", "this afternoon",
        "this evening", "tonight", "just found out", "just came back",
        "a few minutes ago", "an hour ago", "this week",
        "i can't breathe", "i don't know what to do right now",
        "i am so scared right now", "help me", "what do i do",
        "happening right now", "going on right now",
        "my parents are fighting", "someone just", "they just",
        "he just", "she just", "i just", "we just",
        "maintenant", "à l'instant", "ce matin", "ce soir", "aujourd'hui",
        "vient de", "je viens de", "il vient de", "elle vient de",
        "daba", "daba hna", "f had lwaqt", "lyoum", "had sbah"
    ]
    chronic_markers = [
        "always", "every day", "every single day", "all the time",
        "constantly", "keeps happening", "never stops", "over and over",
        "again and again", "every time", "every week",
        "for months", "for years", "since forever", "my whole life",
        "as long as i remember", "since i was little", "for a long time",
        "for so long", "forever", "it never changes", "it never gets better",
        "nobody ever", "no one ever", "they always", "he always", "she always",
        "i always", "i never", "i can never", "nobody understands me",
        "no one ever listens", "no one ever cares", "i am always",
        "i feel like this all the time",
        "toujours", "tout le temps", "depuis longtemps", "chaque jour",
        "ça ne change jamais", "personne ne comprend jamais",
        "depuis des mois", "depuis des années",
        "dima", "f kol waqt", "had chi ma ytbdalch",
        "ma kaynch had li", "walo ma tbdal"
    ]

    for m in immediate_markers:
        if m in text_lower:
            immediate_score += 1
    for m in chronic_markers:
        if m in text_lower:
            chronic_score += 1

    if immediate_score == 0 and chronic_score == 0:
        return 2.0
    if immediate_score > chronic_score:
        return 1.0
    elif chronic_score > immediate_score:
        return 3.0
    else:
        return 2.0

print("detect_EI and detect_RT defined.")


# =============================================================================
# CELL 6 — CASS Retrieval Function
# =============================================================================
# Uses ALPHA, BETA1, BETA2, LAM, MU defined in Cell 7.
# Run Cell 7 before running this cell.

THRESHOLD = 0.25 # old value was set to 0.38

def retrieve(query: str, top_k: int = 1, verbose: bool = True):
    # Step 1 — encode query
    query_vec = model.encode([query], normalize_embeddings=True)[0]

    # Step 2 — cosine similarity
    cos_scores = corpus_embeddings @ query_vec

    # Step 3 — detect EI and RT
    ei_q = detect_EI(query)
    rt_q = detect_RT(query)

    if verbose:
        print(f"Query: '{query}'")
        print(f"Detected EI: {ei_q:.1f} | Detected RT: {rt_q:.1f}")
        print()

    # Step 4 — compute CASS for each entry
    cass_scores = []
    for i, entry in enumerate(corpus):
        cos   = float(cos_scores[i])
        c_aea = C_AEA(ei_q, entry["EI"], lam=LAM)
        c_res = C_RES(rt_q, entry["RT"],  mu=MU)
        cass  = ALPHA * cos + BETA1 * c_aea + BETA2 * c_res
        cass_scores.append((entry, cass, cos, c_aea, c_res))

    # Step 5 — sort by CASS score
    cass_scores.sort(key=lambda x: x[1], reverse=True)

    if verbose:
        print(f"{'Rank':<5} {'ID':<22} {'Prophet':<12} {'CASS':>6} {'cos':>6} {'C_AEA':>6} {'C_RES':>6}")
        print("-" * 70)
        for rank, (entry, cass, cos, c_aea, c_res) in enumerate(cass_scores[:10], 1):
            print(f"{rank:<5} {entry['id']:<22} {entry['prophet_latin']:<12} "
                  f"{cass:>6.3f} {cos:>6.3f} {c_aea:>6.3f} {c_res:>6.3f}")
        print()

    # Step 6 — threshold check
    best_entry, best_score, *_ = cass_scores[0]
    if best_score < THRESHOLD:
        if verbose:
            print(f"Score {best_score:.3f} below threshold. Ask child to rephrase.")
        return []

    # Step 7 — return top_k
    results = [(entry, score) for entry, score, *_ in cass_scores[:top_k]]

    if verbose:
        entry, score = results[0]
        print(f"TOP MATCH: {entry['id']} — {entry['prophet_latin']}")
        print(f"Score: {score:.3f}")
        print(f"Situation: {entry['situation_en'][:120]}...")
        print(f"Child paraphrase: {entry['child_paraphrase'][:120]}...")

    return results

print("retrieve() defined.")


# =============================================================================
# CELL 7 — Optimized parameters from Dirichlet weight search
# =============================================================================
# Best configuration found across 4000 configurations (n=200 Dirichlet
# samples × 20 λ/μ pairs). MHR = 1.60 / 3.000 on 20-query validation set.

# CELL 7 — Optimized parameters (QESC v1.3, 25-query validation set)
# MHR = 1.739 / 3.000
ALPHA = 0.7239
BETA1 = 0.1643
BETA2 = 0.1119
LAM   = 0.2
MU    = 2.0
THRESHOLD = 0.38

print(f"Parameters set: α={ALPHA}, β1={BETA1}, β2={BETA2}, λ={LAM}, μ={MU}")
print(f"Check α+β1+β2 = {ALPHA+BETA1+BETA2:.6f}")

# =============================================================================
# CELL 8 — Demo: run 7 test queries
# =============================================================================

test_queries = [
    "My classmate laughed at me and I feel embarrassed",
    "My friend left me and I feel very sad and alone",
    "I made a big mistake and I am so scared of what will happen",
    "Nobody listens to me no matter what I do",
    "My parents are fighting right now and I am really scared",
    "Je me sens abandonné par mes amis",
    "ma kaynch had li yfahmni",
]

for query in test_queries:
    print("=" * 70)
    results = retrieve(query, top_k=1, verbose=True)
    print()


In [ ]:
# =============================================================================
# CELL 9 — Dirichlet Weight Search (QESC v1.7, 23-query validation set)
# =============================================================================
# Updated validation set: 23 queries covering all major emotional themes.
# EI distribution: EI=1×3, EI=2×6, EI=3×7, EI=4×4, EI=5×3
# RT distribution: RT=1×5, RT=2×12, RT=3×6
# Theme coverage: 23/25 themes (92%) — sibling jealousy, identity, helping
#                 others, parental conflict, and multilingual all covered.
#
# Requires: corpus, corpus_embeddings, model from Cell 3
# Output: best_cass_params.json saved to Drive
# Runtime: ~3 minutes on Colab CPU
# =============================================================================

import itertools
import numpy as np
import json

# ── SEARCH CONFIG ─────────────────────────────────────────────────────────────
N_WEIGHT_SAMPLES = 200
RANDOM_SEED      = 42
LAMBDA_VALUES    = [0.2, 0.5, 1.0, 1.5, 2.0, 2.5]
MU_VALUES        = [0.5, 1.0, 1.5, 2.0]

# ── VALIDATION SET (23 queries) ───────────────────────────────────────────────
VALIDATION_SET = [
    # Q1 — EI=1 mild embarrassment
    ("I feel a little embarrassed about something small today",
     "muhammad_026", ["noah_005", "muhammad_018"]),
    # Q2 — EI=1 mild sadness
    ("I was a bit sad today but I think it will be okay",
     "david_002", ["noah_004", "solomon_003"]),
    # Q3 — EI=1.5 fear of criticism
    ("I am scared to do anything because I am afraid of being told off",
     "muhammad_019", ["moses_002", "david_001"]),
    # Q4 — EI=2.5 public embarrassment
    ("My classmate laughed at me and I feel embarrassed",
     "hud_002", ["noah_005", "moses_007"]),
    # Q5 — EI=2.5 French abandoned by friends
    ("Je me sens abandonné par mes amis",
     "muhammad_002", ["joseph_003", "jacob_002"]),
    # Q6 — EI=2 judge between friends
    ("My friends asked me to decide who is right in their argument",
     "david_004", ["muhammad_013", "solomon_005"]),
    # Q7 — EI=2 social exclusion
    ("I feel left out when my friends play without me",
     "noah_005", ["elijah_001", "muhammad_009"]),
    # Q8 — EI=2 friend argument
    ("I had an argument with my best friend and now we are not talking",
     "muhammad_010", ["muhammad_013", "david_004"]),
    # Q9 — EI=3 friend left, sad alone
    ("My friend left me and I feel very sad and alone",
     "joseph_003", ["muhammad_002", "jacob_001"]),
    # Q10 — EI=3 Darija nobody understands
    ("ma kaynch had li yfahmni",
     "shuaib_001", ["hud_001", "muhammad_006"]),
    # Q11 — EI=3 chronic loneliness
    ("I always feel alone even when I am with people",
     "elijah_001", ["muhammad_002", "jonah_002"]),
    # Q12 — EI=3 want to forgive
    ("Someone hurt me badly but I want to forgive them",
     "joseph_007", ["muhammad_004", "joseph_004"]),
    # Q13 — EI=3.5 blamed unfairly
    ("Everyone blamed me for something I did not do",
     "aaron_001", ["joseph_002", "mary_002"]),
    # Q14 — EI=3 sibling jealousy (NEW)
    ("My brother always gets more attention than me and it is not fair",
     "adam_003", ["joseph_001", "jacob_003"]),
    # Q15 — EI=3 identity / feeling different (NEW)
    ("I always feel different from everyone and nobody understands me",
     "muhammad_006", ["muhammad_020", "elijah_001"]),
    # Q16 — EI=4 big mistake scared
    ("I made a big mistake and I am so scared of what will happen",
     "muhammad_017", ["moses_001", "aaron_001"]),
    # Q17 — EI=4 parents fighting acute
    ("My parents are fighting right now and I am really scared",
     "moses_003", ["muhammad_014", "lot_002"]),
    # Q18 — EI=4 secret betrayal
    ("I just found out my best friend told everyone my secret",
     "muhammad_008", ["joseph_002", "lot_002"]),
    # Q19 — EI=4 fighting at home
    ("There is fighting at home between my parents and I feel scared",
     "moses_003", ["muhammad_014", "lot_001"]),
    # Q20 — EI=5 complete hopelessness
    ("I feel completely hopeless and nothing will ever get better",
     "job_001", ["job_002", "muhammad_001"]),
    # Q21 — EI=5 Allah forgotten me
    ("I feel like Allah has completely forgotten me",
     "muhammad_001", ["job_001", "jonah_001"]),
    # Q22 — EI=5 lost everything broken
    ("I lost everything at once and I feel completely broken",
     "muhammad_016", ["muhammad_007", "job_001"]),
    # Q23 — EI=2 want to help (NEW)
    ("I want to help my friend but I do not know how and I have nothing",
     "muhammad_021", ["joseph_006", "moses_001"]),
]

VAL_EI = [
    1.0,  # Q1
    1.0,  # Q2
    1.5,  # Q3
    2.5,  # Q4
    2.5,  # Q5
    2.0,  # Q6
    2.0,  # Q7
    2.0,  # Q8
    3.0,  # Q9
    3.0,  # Q10
    3.0,  # Q11
    3.0,  # Q12
    3.5,  # Q13
    3.0,  # Q14
    3.0,  # Q15
    4.0,  # Q16
    4.0,  # Q17
    4.0,  # Q18
    4.0,  # Q19
    5.0,  # Q20
    5.0,  # Q21
    5.0,  # Q22
    2.0,  # Q23
]

VAL_RT = [
    2.0,  # Q1
    2.0,  # Q2
    2.0,  # Q3
    2.0,  # Q4
    2.0,  # Q5
    2.0,  # Q6
    2.0,  # Q7
    2.0,  # Q8
    1.0,  # Q9
    3.0,  # Q10
    3.0,  # Q11
    2.0,  # Q12
    2.0,  # Q13
    3.0,  # Q14
    3.0,  # Q15
    1.0,  # Q16
    1.0,  # Q17
    1.0,  # Q18
    1.0,  # Q19
    3.0,  # Q20
    3.0,  # Q21
    2.0,  # Q22
    2.0,  # Q23
]

assert len(VAL_EI) == len(VALIDATION_SET) == 23
assert len(VAL_RT) == len(VALIDATION_SET) == 23

# ── PRE-COMPUTE ───────────────────────────────────────────────────────────────
print(f"Encoding {len(VALIDATION_SET)} validation queries...")
val_queries = [v[0] for v in VALIDATION_SET]
val_best    = [v[1] for v in VALIDATION_SET]
val_alts    = [v[2] for v in VALIDATION_SET]

val_query_vecs = model.encode(val_queries, normalize_embeddings=True)
cos_matrix     = val_query_vecs @ corpus_embeddings.T

ei_array = np.array([e['EI'] for e in corpus], dtype=float)
rt_array = np.array([e['RT'] for e in corpus], dtype=float)

print(f"  Done. Cosine matrix: {cos_matrix.shape}")
print(f"  Corpus size: {len(corpus)} | Validation: {len(VALIDATION_SET)} queries")
print()

# ── SCORING ───────────────────────────────────────────────────────────────────
def compute_mhr(alpha, beta1, beta2, lam, mu):
    scores = []
    for i in range(len(VALIDATION_SET)):
        cos   = cos_matrix[i]
        c_aea = np.exp(-lam * (VAL_EI[i] - ei_array) ** 2)
        c_res = np.exp(-mu  * (VAL_RT[i] - rt_array) ** 2)
        cass  = alpha * cos + beta1 * c_aea + beta2 * c_res
        top_id = corpus[int(np.argmax(cass))]['id']
        if top_id == val_best[i]:        scores.append(3)
        elif top_id in val_alts[i]:      scores.append(2)
        else:                            scores.append(1)
    return float(np.mean(scores))

# ── SEARCH ────────────────────────────────────────────────────────────────────
np.random.seed(RANDOM_SEED)
all_results = []

lam_mu_pairs  = list(itertools.product(LAMBDA_VALUES, MU_VALUES))
total_configs = len(lam_mu_pairs) * N_WEIGHT_SAMPLES

print(f"Starting search: {total_configs} configurations...")
print(f"  {len(lam_mu_pairs)} (λ,μ) pairs × {N_WEIGHT_SAMPLES} Dirichlet samples")
print()

best_so_far = -1.0
for pair_idx, (lam, mu) in enumerate(lam_mu_pairs):
    weight_samples = np.random.dirichlet([1, 1, 1], size=N_WEIGHT_SAMPLES)
    for alpha, beta1, beta2 in weight_samples:
        mhr = compute_mhr(alpha, beta1, beta2, lam, mu)
        all_results.append((mhr, alpha, beta1, beta2, lam, mu))
        if mhr > best_so_far:
            best_so_far = mhr
    if (pair_idx + 1) % 4 == 0 or pair_idx == 0:
        print(f"  [{pair_idx+1:>2}/{len(lam_mu_pairs)}] λ={lam:.1f}, μ={mu:.1f} "
              f"| best MHR so far: {best_so_far:.3f}")

# ── SELECT BEST ───────────────────────────────────────────────────────────────
all_results.sort(reverse=True)
max_mhr  = all_results[0][0]
optimal  = [(mhr,a,b1,b2,l,m) for mhr,a,b1,b2,l,m in all_results if mhr==max_mhr]
# Among tied: select lowest α (most constraint-dominant)
best     = min(optimal, key=lambda x: x[1])
best_mhr = best[0]
alpha, beta1, beta2, lam, mu = best[1], best[2], best[3], best[4], best[5]

# ── RESULTS ───────────────────────────────────────────────────────────────────
print()
print("=" * 60)
print("SEARCH COMPLETE")
print("=" * 60)
print(f"\nBest configuration (lowest α among {len(optimal)} tied at MHR={max_mhr:.3f}):")
print(f"  α  = {alpha:.4f}  (cosine)")
print(f"  β1 = {beta1:.4f}  (C_AEA)")
print(f"  β2 = {beta2:.4f}  (C_RES)")
print(f"  λ  = {lam:.1f}    (C_AEA sharpness)")
print(f"  μ  = {mu:.1f}    (C_RES sharpness)")
print(f"  MHR = {best_mhr:.3f} / 3.000")
print(f"  Check α+β1+β2 = {alpha+beta1+beta2:.6f}")

all_results.sort(reverse=True)
print()
print("Top 10 configurations:")
print(f"{'Rank':<5} {'MHR':>5} {'α':>7} {'β1':>7} {'β2':>7} {'λ':>5} {'μ':>5}")
print("-" * 50)
seen = set()
rank = 0
for mhr, a, b1, b2, l, m in all_results:
    key = (round(a,3), round(b1,3), round(b2,3), l, m)
    if key in seen: continue
    seen.add(key)
    rank += 1
    print(f"{rank:<5} {mhr:>5.3f} {a:>7.3f} {b1:>7.3f} {b2:>7.3f} {l:>5.1f} {m:>5.1f}")
    if rank == 10: break

mhr_vals = np.array([r[0] for r in all_results])
print()
print(f"MHR: min={mhr_vals.min():.3f} max={mhr_vals.max():.3f} "
      f"mean={mhr_vals.mean():.3f} std={mhr_vals.std():.3f}")

print()
print("Convergence check:")
for n in [20, 40, 60, 80, 100, 150, 200]:
    best_at_n = []
    for pidx, (lam_v, mu_v) in enumerate(lam_mu_pairs):
        start  = pidx * N_WEIGHT_SAMPLES
        subset = all_results[start:start+n]
        if subset: best_at_n.append(max(r[0] for r in subset))
    if best_at_n:
        print(f"  n={n:>3}: {np.mean(best_at_n):.3f}")

best_params = {
    'alpha': float(alpha), 'beta1': float(beta1), 'beta2': float(beta2),
    'lambda': float(lam), 'mu': float(mu), 'mhr': float(best_mhr),
    'n_configs_searched': total_configs,
    'n_validation_queries': len(VALIDATION_SET),
    'corpus_version': 'v1.7', 'corpus_size': len(corpus),
    'selection_criterion': 'lowest alpha among tied MHR configurations',
}
with open('best_cass_params.json', 'w') as f:
    json.dump(best_params, f, indent=2)
print()
print("Saved best_cass_params.json")
print("Update ALPHA, BETA1, BETA2, LAM, MU in Cell 7 with these values.")

In [ ]:
# =============================================================================
# CELL 9 — Top 10 comparison: Cosine-only vs CASS
# Shows C_AEA safety gate effect on emotionally dangerous candidates
# =============================================================================

test_queries = [
    ("I feel a little embarrassed today",              1.5, 2.0),
    ("My classmate laughed at me I feel embarrassed",  2.5, 2.0),
    ("Nobody listens to me no matter what I do",       2.0, 2.0),
    ("I feel happy and grateful today",                1.0, 1.0),
    ("ma kaynch had li yfahmni",                       3.0, 3.0),
]

print("C_AEA SAFETY GATE DEMONSTRATION")
print("Shows dangerous entries suppressed by CASS vs cosine-only ranking")

for query, ei_q, rt_q in test_queries:
    qvec = model.encode([query], normalize_embeddings=True)[0]
    cos_scores = corpus_embeddings @ qvec

    results = []
    for i, entry in enumerate(corpus):
        cos   = float(cos_scores[i])
        c_aea = C_AEA(ei_q, entry['EI'], lam=LAM)
        c_res = C_RES(rt_q, entry['RT'],  mu=MU)
        cass  = ALPHA * cos + BETA1 * c_aea + BETA2 * c_res
        results.append((entry, cass, cos, c_aea, c_res))

    cass_sorted = sorted(results, key=lambda x: x[1], reverse=True)
    cos_sorted  = sorted(results, key=lambda x: x[2], reverse=True)

    # Find entries with dangerous EI mismatch (≥2 levels) in top 15 cosine
    dangerous = []
    for rank, (entry, cass, cos, c_aea, c_res) in enumerate(cos_sorted[:15]):
        if abs(entry['EI'] - ei_q) >= 2:
            cass_rank = next(
                (r+1 for r, (e,_,_,_,_) in enumerate(cass_sorted) if e['id']==entry['id']), 99
            )
            dangerous.append((rank+1, cass_rank, entry, cos, c_aea, cass))

    if dangerous:
        print(f"\n{'='*70}")
        print(f"Query: '{query}' | EI={ei_q}")
        print(f"{'Entry':<22} {'EI':>3} {'cos':>6} {'C_AEA':>6} {'cos_rank':>9} {'cass_rank':>10} {'effect'}")
        print("-"*70)
        for cos_rank, cass_rank, entry, cos, c_aea, cass in dangerous:
            drop = cass_rank - cos_rank
            effect = f"↓ {drop} ranks" if drop > 0 else "unchanged"
            print(f"{entry['id']:<22} {entry['EI']:>3} {cos:>6.3f} {c_aea:>6.3f} {cos_rank:>9} {cass_rank:>10}  {effect}")

In [ ]:
# =============================================================================
# CELL 10 — C_RES Safety Gate Demonstration
# Shows resolution-type dangerous entries suppressed by CASS vs cosine-only
# Dangerous = |RT(q) - RT(c)| >= 2
# i.e. acute query (RT=1) matched to delayed-resolution entry (RT=3)
# or chronic query (RT=3) matched to immediate-resolution entry (RT=1)
# =============================================================================

cres_queries = [
    # Acute crisis queries (RT=1) — should NOT get RT=3 delayed stories
    ("My parents are fighting right now and I am really scared",   4.0, 1.0),
    ("I just found out my best friend told everyone my secret",    4.0, 1.0),
    ("Something very scary is happening right now help me",        4.5, 1.0),
    # Chronic situation queries (RT=3) — should NOT get RT=1 immediate stories
    ("I have been waiting for so long and nothing is changing",    4.0, 3.0),
    ("Every single day I feel like this and it never gets better", 3.0, 3.0),
]

print("C_RES SAFETY GATE DEMONSTRATION")
print("Shows RT-mismatched entries suppressed by CASS vs cosine-only ranking")
print("Dangerous threshold: |RT(q) - RT(c)| >= 2")
print()

found_any = False

for query, ei_q, rt_q in cres_queries:
    qvec = model.encode([query], normalize_embeddings=True)[0]
    cos_scores = corpus_embeddings @ qvec

    results = []
    for i, entry in enumerate(corpus):
        cos   = float(cos_scores[i])
        c_aea = C_AEA(ei_q, entry['EI'], lam=LAM)
        c_res = C_RES(rt_q, entry['RT'],  mu=MU)
        cass  = ALPHA * cos + BETA1 * c_aea + BETA2 * c_res
        results.append((entry, cass, cos, c_aea, c_res))

    cass_sorted = sorted(results, key=lambda x: x[1], reverse=True)
    cos_sorted  = sorted(results, key=lambda x: x[2], reverse=True)

    # Find dangerous entries: |RT(q) - RT(c)| >= 2 in top 20 cosine results
    dangerous = []
    for rank, (entry, cass, cos, c_aea, c_res) in enumerate(cos_sorted[:20]):
        if abs(entry['RT'] - rt_q) >= 2:
            cass_rank = next(
                (r+1 for r, (e,_,_,_,_) in enumerate(cass_sorted)
                 if e['id'] == entry['id']), 99
            )
            dangerous.append({
                'cos_rank': rank + 1,
                'cass_rank': cass_rank,
                'entry': entry,
                'cos': cos,
                'c_aea': c_aea,
                'c_res': c_res,
                'cass': cass,
                'drop': cass_rank - (rank + 1)
            })

    if dangerous:
        found_any = True
        rt_label = "immediate" if rt_q == 1.0 else "chronic"
        print(f"{'='*75}")
        print(f"Query: '{query}'")
        print(f"EI={ei_q} | RT={rt_q} ({rt_label}) | "
              f"Dangerous RT_c = {'3 (delayed)' if rt_q==1.0 else '1 (immediate)'}")
        print()
        print(f"{'Entry':<22} {'RT_c':>5} {'cos':>6} {'C_RES':>6} "
              f"{'C_AEA':>6} {'cos_rank':>9} {'cass_rank':>10} {'effect'}")
        print("-"*80)
        for d in dangerous:
            rt_label_c = "delayed" if d['entry']['RT'] == 3 else "immediate"
            drop_str = f"↓ {d['drop']} ranks" if d['drop'] > 0 else "unchanged"
            print(f"{d['entry']['id']:<22} {d['entry']['RT']:>5} "
                  f"{d['cos']:>6.3f} {d['c_res']:>6.3f} {d['c_aea']:>6.3f} "
                  f"{d['cos_rank']:>9} {d['cass_rank']:>10}  {drop_str}")
        print()

        # Also show what CASS correctly returns instead
        top_cass = cass_sorted[0]
        print(f"CASS top match: {top_cass[0]['id']} "
              f"(EI={top_cass[0]['EI']}, RT={top_cass[0]['RT']}) "
              f"— CASS={top_cass[1]:.3f}, cos={top_cass[2]:.3f}, "
              f"C_RES={top_cass[4]:.3f}")
        print()

if not found_any:
    print("No dangerous RT entries found in cosine top 20 for these queries.")
    print("This means cosine is already selecting RT-appropriate candidates.")
    print("Interpret as: C_RES operates as a silent safety gate (no dangerous")
    print("candidates to suppress) — consistent with C_AEA behavior in normal operation.")

In [ ]:
# =============================================================================
# CELL 11 — TF-IDF + BM25 Baselines + Ablation Study: Instantiation 1 (QESC)
# STANDALONE — does not require any previous cell to have been run
#              except: Cell 3 (model) and corpus JSON file on Drive
#
# What this cell does:
#   1. Loads QESC corpus from JSON
#   2. Encodes 23 validation queries using MiniLM
#   3. Evaluates TF-IDF baseline (lexical, simple word-frequency weighting)
#   4. Evaluates BM25 baseline (lexical, saturation + length normalization)
#   5. Evaluates 3 CASS ablation variants (cosine-only, C_AEA only, C_RES only)
#   6. Evaluates Full CASS (α=0.724, β1=0.164, β2=0.112, λ=0.2, μ=2.0)
#   7. Prints complete ablation table with MHR and P@1
#
# Baselines comparison:
#   TF-IDF — simplest lexical baseline, no saturation or length correction
#   BM25   — stronger lexical baseline, adds saturation + length normalization
#   Both fail on Darija/French queries and indirect emotional expressions
#   because they require exact word overlap with corpus descriptions
#
# Expected result: TF-IDF ≤ BM25 < Cosine < CASS variants < Full CASS
#
# Requires: model (from Cell 3), QESC_v1.0.json (on Google Drive)
# Runtime:  < 30 seconds on CPU
# =============================================================================

import json
with open('QESC_v1.0.json', encoding='utf-8') as f:
    corpus = json.load(f)

#── PARAMETERS ────────────────────────────────────────────────────────────────
A, B1, B2, L, M = 0.7239, 0.1643, 0.1119, 0.2, 2.0

# ── VALIDATION SET (23 queries — copied from Cell 6) ─────────────────────────
VALIDATION_SET = [
    # EI=1
    ("I feel a little embarrassed about something small today",
     "muhammad_026", ["noah_005", "muhammad_018"]),
    ("I was a bit sad today but I think it will be okay",
     "david_002", ["noah_004", "solomon_003"]),
    ("I am scared to do anything because I am afraid of being told off",
     "muhammad_019", ["moses_002", "david_001"]),
    # EI=2
    ("My classmate laughed at me and I feel embarrassed",
     "hud_002", ["noah_005", "moses_007"]),
    ("Je me sens abandonné par mes amis",
     "muhammad_002", ["joseph_003", "jacob_002"]),
    ("My friends asked me to decide who is right in their argument",
     "david_004", ["muhammad_013", "solomon_005"]),
    ("I feel left out when my friends play without me",
     "noah_005", ["elijah_001", "muhammad_009"]),
    ("I had an argument with my best friend and now we are not talking",
     "muhammad_010", ["muhammad_013", "david_004"]),
    # EI=3
    ("My friend left me and I feel very sad and alone",
     "joseph_003", ["muhammad_002", "jacob_001"]),
    ("ma kaynch had li yfahmni",
     "shuaib_001", ["hud_001", "muhammad_006"]),
    ("I always feel alone even when I am with people",
     "elijah_001", ["muhammad_002", "jonah_002"]),
    ("Someone hurt me badly but I want to forgive them",
     "joseph_007", ["muhammad_004", "joseph_004"]),
    ("Everyone blamed me for something I did not do",
     "aaron_001", ["joseph_002", "mary_002"]),
    ("My brother always gets more attention than me and it is not fair",
     "adam_003", ["joseph_001", "jacob_003"]),
    ("I always feel different from everyone and nobody understands me",
     "muhammad_006", ["muhammad_020", "elijah_001"]),
    # EI=4
    ("I made a big mistake and I am so scared of what will happen",
     "muhammad_017", ["moses_001", "aaron_001"]),
    ("My parents are fighting right now and I am really scared",
     "moses_003", ["muhammad_014", "lot_002"]),
    ("I just found out my best friend told everyone my secret",
     "muhammad_008", ["joseph_002", "lot_002"]),
    ("There is fighting at home between my parents and I feel scared",
     "moses_003", ["muhammad_014", "lot_001"]),
    # EI=5
    ("I feel completely hopeless and nothing will ever get better",
     "job_001", ["job_002", "muhammad_001"]),
    ("I feel like Allah has completely forgotten me",
     "muhammad_001", ["job_001", "jonah_001"]),
    ("I lost everything at once and I feel completely broken",
     "muhammad_016", ["muhammad_007", "job_001"]),
    # EI=2 helping others
    ("I want to help my friend but I do not know how and I have nothing",
     "muhammad_021", ["joseph_006", "moses_001"]),
]

VAL_EI = [
    1.0, 1.0, 1.5,
    2.5, 2.5, 2.0, 2.0, 2.0,
    3.0, 3.0, 3.0, 3.0, 3.5, 3.0, 3.0,
    4.0, 4.0, 4.0, 4.0,
    5.0, 5.0, 5.0,
    2.0,
]

VAL_RT = [
    2.0, 2.0, 2.0,
    2.0, 2.0, 2.0, 2.0, 2.0,
    1.0, 3.0, 3.0, 2.0, 2.0, 3.0, 3.0,
    1.0, 1.0, 1.0, 1.0,
    3.0, 3.0, 2.0,
    2.0,
]

val_best = [v[1] for v in VALIDATION_SET]
val_alts = [v[2] for v in VALIDATION_SET]

assert len(VAL_EI) == len(VALIDATION_SET) == 23
assert len(VAL_RT) == len(VALIDATION_SET) == 23

# ── ENCODE VALIDATION QUERIES ─────────────────────────────────────────────────
print(f"Encoding {len(VALIDATION_SET)} validation queries...")
val_vecs   = model.encode(
    [v[0] for v in VALIDATION_SET],
    normalize_embeddings=True
)
cos_matrix = val_vecs @ corpus_embeddings.T
ei_array   = np.array([e['EI'] for e in corpus], dtype=float)
rt_array   = np.array([e['RT'] for e in corpus], dtype=float)
print(f"Cosine matrix: {cos_matrix.shape}")
print()


In [ ]:
# =============================================================================
# CELL 12 — BM25 Baseline + Ablation Study: Instantiation 1 (QESC)
# STANDALONE — does not require Cell 6 (weight search) to have been run
#
# Requires only: corpus, corpus_embeddings, model from Cells 3-4
# =============================================================================

# Install
!pip install rank_bm25 -q
import numpy as np
from rank_bm25 import BM25Okapi
# If not installed: !pip install rank_bm25 -q

#
# ── BM25 BASELINE ─────────────────────────────────────────────────────────────
# Keyword matching on prophet situation descriptions.
# No semantic understanding — fails completely on this corpus.

tokenized_corpus = [e['situation_en'].lower().split() for e in corpus]
bm25 = BM25Okapi(tokenized_corpus)

def bm25_score():
    scores = []
    for i in range(len(VALIDATION_SET)):
        q_tokens = VALIDATION_SET[i][0].lower().split()
        bm25_scores = bm25.get_scores(q_tokens)
        top_id = corpus[int(np.argmax(bm25_scores))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    return float(np.mean(scores)), sum(1 for s in scores if s==3)/len(scores)

# ── CASS VARIANTS ─────────────────────────────────────────────────────────────

def score_variant(alpha, beta1, beta2, lam=L, mu=M):
    scores = []
    for i in range(len(VALIDATION_SET)):
        cos   = cos_matrix[i]
        c_aea = np.exp(-lam * (VAL_EI[i] - ei_array) ** 2)
        c_res = np.exp(-mu  * (VAL_RT[i] - rt_array) ** 2)
        cass  = alpha * cos + beta1 * c_aea + beta2 * c_res
        top_id = corpus[int(np.argmax(cass))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    return float(np.mean(scores)), sum(1 for s in scores if s==3)/len(scores)

# ── RUN ALL VARIANTS ──────────────────────────────────────────────────────────

print("=" * 55)
print("ABLATION STUDY — INSTANTIATION 1 (QESC)")
print(f"Validation set: {len(VALIDATION_SET)} queries")
print("=" * 55)
print()
print(f"{'System':<32} {'MHR':>6} {'P@1':>6}")
print("-" * 46)

variants = [
    ("BM25 baseline",                None),
    ("Cosine-only (α=1)",            (1.0,  0.0,      0.0   )),
    ("CASS + C_AEA only (β₂=0)",     (A,    B1+B2,    0.0   )),
    ("CASS + C_RES only (β₁=0)",     (A,    0.0,      B1+B2 )),
    ("Full CASS (C_AEA + C_RES)",    (A,    B1,       B2    )),
]

all_results = {}
for name, params in variants:
    if params is None:
        mhr, p1 = bm25_score()
    else:
        mhr, p1 = score_variant(*params)
    all_results[name] = (mhr, p1)
    marker = " ←" if name == "Full CASS (C_AEA + C_RES)" else ""
    print(f"{name:<32} {mhr:>6.3f} {p1:>6.2f}{marker}")

print()
print("Key findings:")
bm25_mhr = all_results["BM25 baseline"][0]
cos_mhr  = all_results["Cosine-only (α=1)"][0]
cass_mhr = all_results["Full CASS (C_AEA + C_RES)"][0]
cass_p1  = all_results["Full CASS (C_AEA + C_RES)"][1]

print(f"  Full CASS vs BM25:         ΔMHR={cass_mhr-bm25_mhr:+.3f}")
print(f"  Full CASS vs cosine-only:  ΔMHR={cass_mhr-cos_mhr:+.3f}")
print(f"  C_AEA contribution:        ΔMHR={all_results['CASS + C_AEA only (β₂=0)'][0]-cos_mhr:+.3f}")
print(f"  C_RES contribution:        ΔMHR={all_results['CASS + C_RES only (β₁=0)'][0]-cos_mhr:+.3f}")
print(f"  Synergy (both > each):     {cass_mhr:.3f} > {all_results['CASS + C_AEA only (β₂=0)'][0]:.3f} and {all_results['CASS + C_RES only (β₁=0)'][0]:.3f}")


In [ ]:
# =============================================================================
# CELL 13 — TF-IDF Baseline: Instantiation 1 (QESC)
# =============================================================================
# Computes TF-IDF retrieval score on the 23-query QESC validation set.
# TF-IDF indexes prophet situation descriptions and ranks by word-frequency
# similarity to child queries.
# Unlike BM25, TF-IDF does not apply saturation or document length
# normalization — making it a simpler but weaker lexical baseline.
# Expected to fail completely on Darija/French queries (vocabulary mismatch)
# and indirect emotional expressions (no shared keywords with corpus).
# Expected result: TF-IDF ≤ BM25 << Cosine < Full CASS
#
# Requires: corpus, VALIDATION_SET, val_best, val_alts from previous cells
# Runtime: <5 seconds on CPU
# =============================================================================
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Build TF-IDF index on corpus descriptions
corpus_texts = [e['situation_en'] for e in corpus]  # or e['description'] for PhonEx
vectorizer = TfidfVectorizer()
tfidf_matrix = vectorizer.fit_transform(corpus_texts)

def tfidf_score():
    scores = []
    for i in range(len(VALIDATION_SET)):
        q_vec = vectorizer.transform([VALIDATION_SET[i][0]])
        sims = cosine_similarity(q_vec, tfidf_matrix).flatten()
        top_id = corpus[int(np.argmax(sims))]['id']
        if top_id == val_best[i]:      scores.append(3)
        elif top_id in val_alts[i]:    scores.append(2)
        else:                          scores.append(1)
    mhr = float(np.mean(scores))
    p1  = sum(1 for s in scores if s==3)/len(scores)
    return mhr, p1

mhr_tfidf, p1_tfidf = tfidf_score()
print(f"TF-IDF baseline: MHR={mhr_tfidf:.3f}, P@1={p1_tfidf:.2f}")